In [3]:
from typing import Dict, List, Set, Tuple

Graph = Dict[int, List[int]]

def vertex_cover_pricing(
    vertices: List[int],
    edges: List[Tuple[int, int]],
    weight: Dict[int, float]

) -> Tuple[Set[int], float, Dict[Tuple[int,int], float]]:

    prices : Dict[Tuple[int,int], float] = {e: 0.0 for e in edges}
    paid: Dict[int, float]               = {v: 0.0 for v in vertices}
    cover: Set[int]                      = set()

    def is_tight(v) -> bool:
      return paid[v] >= weight[v] - 1e-9

    def is_covered(u: int, v: int) -> bool:
      return u in cover or v in cover

    uncovered = [e for e in edges if not is_covered(*e)]

    while uncovered:
      u,v = uncovered.pop(0)

      if is_covered(u,v):
        continue

      slack_u = weight[u] - paid[u]
      slack_v = weight[v] - paid[v]
      delta = min(slack_u, slack_v)

      prices[(u,v)] += delta
      paid[u]       += delta
      paid[v]       += delta

      for node in (u,v):
        if is_tight(node):
          cover.add(node)

      uncovered = [e for e in edges if not is_covered(*e)]


    cost = sum(weight[v] for v in cover)
    return cover, cost, prices


def lower_bound_opt(
    edges: List[Tuple[int, int]],
    prices: Dict[Tuple[int,int], float]
) -> float:
     return sum(prices.values())



if __name__ == "__main__":
    #  1 — 2
    #  |\ |
    #  | \|
    #  3 — 4 — 5
    vertices = [1, 2, 3, 4, 5]
    edges    = [(1,2), (1,3), (1,4), (2,4), (3,4), (4,5)]
    weight   = {1: 3.0, 2: 2.0, 3: 2.0, 4: 4.0, 5: 1.0}

    cover, cost, prices = vertex_cover_pricing(vertices, edges, weight)
    lb = lower_bound_opt(edges, prices)

    print("Grafo:")
    print(f"  Vértices : {vertices}")
    print(f"  Aristas  : {edges}")
    print(f"  Pesos    : {weight}\n")

    print(f"Precios asignados a aristas:")
    for e, p in prices.items():
        print(f"  {e}: {p:.2f}")

    print(f"\nCobertura encontrada : {sorted(cover)}")
    print(f"Costo de la cobertura: {cost:.2f}")
    print(f"Cota inferior OPT    : {lb:.2f}")
    print(f"Ratio aproximación   : {cost/lb:.4f}")

Grafo:
  Vértices : [1, 2, 3, 4, 5]
  Aristas  : [(1, 2), (1, 3), (1, 4), (2, 4), (3, 4), (4, 5)]
  Pesos    : {1: 3.0, 2: 2.0, 3: 2.0, 4: 4.0, 5: 1.0}

Precios asignados a aristas:
  (1, 2): 2.00
  (1, 3): 1.00
  (1, 4): 0.00
  (2, 4): 0.00
  (3, 4): 1.00
  (4, 5): 1.00

Cobertura encontrada : [1, 2, 3, 5]
Costo de la cobertura: 8.00
Cota inferior OPT    : 5.00
Ratio aproximación   : 1.6000
